# Dispatch baseline test (rule-based, no optimization)

In [ ]:
%load_ext autoreload
%autoreload 2

from bess_office.fetch_pv import get_pv_profile
from bess_office.fetch_load import get_load_profile
from bess_office.dispatch_sim import run_baseline_dispatch, BatterySpec

## Fetch and align data

In [ ]:
pv = get_pv_profile(latitude=52.01, longitude=4.36, peak_power_kw=10)
load = get_load_profile(peak_kwh_per_quarter=15)
load_hourly = load.resample("1h").sum()  # match PV's hourly resolution

## Run the baseline dispatch

In [ ]:
battery = BatterySpec(capacity_kwh=10, power_kw=10)
result = run_baseline_dispatch(
    pv, load_hourly, battery,
    peak_threshold_kw=8,
    timestep_hours=1.0,
)
result.head()

## Peak shaving result

In [ ]:
naive_import = (result["load_kwh"] - result["pv_kwh"]).clip(lower=0)
peak_shaved = naive_import.max() - result["grid_import_kwh"].max()
print(f"Peak without battery: {naive_import.max():.2f} kW")
print(f"Peak with battery: {result['grid_import_kwh'].max():.2f} kW")
print(f"Peak shaved: {peak_shaved:.2f} kW ({100*peak_shaved/naive_import.max():.0f}%)")

## Self-consumption result

In [ ]:
pv_used_onsite = result["pv_kwh"].sum() - result["grid_export_kwh"].sum()
self_consumption_pct = 100 * pv_used_onsite / result["pv_kwh"].sum()
print(f"Self-consumption: {self_consumption_pct:.0f}%")